In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/2026-challenge/sample_submission.csv
/kaggle/input/2026-challenge/train.csv
/kaggle/input/2026-challenge/test.csv


### Load the dataset for the challenge

In [2]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("moanlobago/2026-challenge")

# print("Path to dataset files:", path)

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

In [4]:
#train = pd.read_csv("/kaggle/input/smokers-dataset/train.csv")
#test = pd.read_csv("/kaggle/input/smokers-dataset/test.csv")
#sample_sub = pd.read_csv("/kaggle/input/smokers-dataset/sample_submission.csv")

####test
train = pd.read_csv("/kaggle/input/2026-challenge/train.csv")
test = pd.read_csv("/kaggle/input/2026-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/2026-challenge/sample_submission.csv")

In [5]:
print(f"Train shape:", train.shape)
print(f"Test shape:", test.shape)
print(f"Sample Submission shape:", sample_sub.shape)

Train shape: (15000, 24)
Test shape: (10000, 23)
Sample Submission shape: (10000, 2)


In [6]:
train.head()

,id,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,smoking
0,0,30.0,175.0,75.0,85.0,1.0,1.2,1.0,1.0,127.0,...,55.0,125.0,15.8,1.0,0.9,18.0,17.0,53.0,0.0,1.0
1,1,50.0,155.0,55.0,73.0,1.0,1.2,1.0,1.0,118.0,...,55.0,101.0,13.3,1.0,0.6,16.0,9.0,16.0,0.0,0.0
2,2,30.0,175.0,60.0,72.0,0.8,0.8,1.0,1.0,119.0,...,74.0,93.0,14.7,1.0,0.9,18.0,13.0,22.0,0.0,0.0
3,3,45.0,155.0,55.0,75.5,1.0,1.2,1.0,1.0,90.0,...,55.0,95.0,13.5,1.0,0.8,17.0,8.0,12.0,0.0,0.0
4,4,60.0,155.0,60.0,81.0,0.7,0.7,1.0,1.0,130.0,...,52.0,88.0,13.4,1.0,0.9,22.0,15.0,21.0,0.0,1.0


In [7]:
test.head()

,id,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,triglyceride,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries
0,15000,40.0,175.0,80.0,95.0,1.0,0.8,1.0,1.0,118.0,...,203.0,39.0,118.0,14.8,1.0,0.8,39.0,64.0,48.0,0.0
1,15001,60.0,165.0,55.0,79.0,0.9,1.2,1.0,1.0,130.0,...,82.0,52.0,121.0,15.0,1.0,0.9,21.0,21.0,17.0,0.0
2,15002,45.0,160.0,60.0,73.2,0.9,0.9,1.0,1.0,110.0,...,77.0,64.0,129.0,9.2,1.0,0.7,16.0,10.0,11.0,0.0
3,15003,55.0,165.0,55.0,79.0,0.9,0.8,1.0,1.0,122.0,...,113.0,54.0,105.0,15.4,1.0,0.9,25.0,25.0,27.0,0.0
4,15004,35.0,180.0,85.0,90.0,0.6,0.6,1.0,1.0,116.0,...,274.0,45.0,137.0,16.3,1.0,1.1,27.0,29.0,47.0,0.0


## Notes:
- Tried to address some of the skewed data I found, most of it right skewed. Applied natural log approach.
- Tried out some feature engineering in earlier attempts, these showed consistent improvements for me (when some feature selection is applied later)

In [8]:
def feature_eng(df):

    # Handle skewed data
    skewed_cols = ['AST', 'ALT', 'Gtp', 'triglyceride']
    for col in skewed_cols:
        df[col] = np.log1p(df[col]) # Applies natural log, suitable for right skewed data

    # BMI
    df['BMI'] = df['weight(kg)'] / (df['height(cm)']**2) * 10000

    # Blood pressure
    df['BP'] = df['systolic'] / df['relaxation']

    # Deritis
    df['Deritis'] = df['AST'] / df['ALT']

    return df

In [9]:
train_df = feature_eng(train)
test_df = feature_eng(test)

### Notes
- Used autogluon to automate experiments and find best ensemble quickly

In [10]:
!pip install autogluon -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 6.2 MB/s eta 0:00:0

In [11]:
#xgb, cat and nn
from autogluon.tabular import TabularDataset, TabularPredictor


In [12]:
# Save ID for submission
# Did not delete/drop Target (smoking), it is needed by autogluon
test_ids = test_df['id']

train_df = train_df.drop(['id'], axis=1)
test_df = test_df.drop('id', axis=1)

### Training gluon
- I used some of the bat parameters of earlier experiments, not really proven to be the best (from hyperparameter tuning), but were providing decent results for xgb and catboost respectively, so I maintained them.
- Key goal here was to find the best ensemble (say 0.3*catboost + 0.4*xgb + 0.3*neuralnet), with maybe some bagging experiments done also

In [13]:
from autogluon.tabular import TabularPredictor
import autogluon.core as ag

# --- XGBoost Parameters ---

xgb_params = {
    'n_estimators': 2000,       
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': -1,
    
    # Resource Allocation: 1 GPU per fold
    'ag_args_fit': {'num_gpus': 1} 
}

# --- CatBoost Parameters---
cat_params = {
    'iterations': 2000,
    'learning_rate': 0.05,
    'depth': 6,
    'random_seed': 42,
    'allow_writing_files': False, # Keeps your directory clean
    
    # Force GPU usage for CatBoost
    'ag_args_fit': {'num_gpus': 1} 
}

# --- Neural Net Parameters ---
nn_params = {
    'num_epochs': 100,         
    'learning_rate': 1e-3,    
    'activation': 'swish',    
    'dropout_prob': 0.3,      
    'hidden_size': 128,       
    'num_layers': 3,          
    'use_batchnorm': True, 
    'ag_args_fit': {'num_gpus': 1}
}

# =======================================================
# 2. Construct the Search Space
# =======================================================

ag_args_ensemble = {'fold_fitting_strategy': 'sequential_local'}  ## Parallel was yielding errors
hyperparameters = {
    'XGB': xgb_params,
    'CAT': cat_params,
    'NN_TORCH': nn_params,
}

# Initialize and train models to get best ensemble
predictor = TabularPredictor(
    label='smoking',
    eval_metric='roc_auc',
    problem_type='binary'
)

predictor.fit(
    train_data=train_df,
    presets='best_quality',
    hyperparameters=hyperparameters, 
    num_gpus=2,   
ag_args_ensemble=ag_args_ensemble)

No path specified. Models will be saved in: "AutogluonModels/ag-20260112_084700"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Sep 27 10:16:09 UTC 2025
CPU Count:          4
Pytorch Version:    2.8.0+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 14.74/14.74 GB | GPU 1: 14.74/14.74 GB
Total GPU Memory:   Free: 29.48 GB, Allocated: 0.00 GB, Total: 29.48 GB
GPU Count:          2
Memory Avail:       29.83 GB / 31.35 GB (95.2%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determin

In [14]:
leaderboard = predictor.leaderboard(silent=True)
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.890637,roc_auc,0.180374,210.869518,0.002273,0.162209,2,True,4
1,WeightedEnsemble_L3,0.890628,roc_auc,0.281323,315.786165,0.002322,0.312524,3,True,8
2,XGBoost_BAG_L1,0.889526,roc_auc,0.049141,7.616494,0.049141,7.616494,1,True,2
3,CatBoost_BAG_L2,0.889156,roc_auc,0.206059,260.294444,0.027958,49.587134,2,True,5
4,NeuralNetTorch_BAG_L2,0.888833,roc_auc,0.279001,315.473641,0.100900,104.766332,2,True,7
5,CatBoost_BAG_L1,0.888765,roc_auc,0.032155,100.042111,0.032155,100.042111,1,True,1
6,XGBoost_BAG_L2,0.887761,roc_auc,0.221691,216.797796,0.043590,6.090487,2,True,6
7,NeuralNetTorch_BAG_L1,0.874945,roc_auc,0.096805,103.048704,0.096805,103.048704,1,True,3


In [15]:
pred1 = predictor.predict_proba(test_df)

In [16]:
# predictor.fit_summary()

In [17]:
model_info = predictor.info()['model_info']['WeightedEnsemble_L2']

weights_dict = model_info['children_info']['S1F1']['model_weights']

print("Model Weights in WeightedEnsemble_L2:")
weights_dict

Model Weights in WeightedEnsemble_L2:


{'CatBoost_BAG_L1': np.float64(0.38461538461538464),
 'XGBoost_BAG_L1': np.float64(0.5384615384615384),
 'NeuralNetTorch_BAG_L1': np.float64(0.07692307692307693)}

In [18]:
pred1

,0,1
0,0.240495,0.759505
1,0.616865,0.383135
2,0.992287,0.007713
3,0.308153,0.691847
4,0.266600,0.733400
...,...,...
9995,0.981795,0.018205
9996,0.279320,0.720680
9997,0.896276,0.103724
9998,0.467590,0.532410


### Submission files

In [19]:
submission = pd.DataFrame({'id': test_ids, 'smoking': pred1[1]})

submission.to_csv('submission_reprod.csv', index=False)
print("Submission saved")

Submission saved
